# PaddleOCR Engine (Phase 6)

This notebook is the **source of truth** for the PaddleOCR engine used by the
Employee Document Verification Portal. The Python service in
`app/services/paddleocr/service.py` executes this notebook via `nbclient`
and pulls out a callable that recognizes text in a BGR image.

Open this file in Jupyter/VS Code to tweak the engine, swap language models,
or change preprocessing settings without touching the rest of the codebase.

In [ ]:
# Parameters are injected by the NotebookRunner.
import os, json, time
from pathlib import Path

LANG = (parameters or {}).get("lang", "en")
USE_ANGLE_CLS = bool((parameters or {}).get("use_angle_cls", True))
DET_MODEL_DIR = (parameters or {}).get("det_model_dir")
REC_MODEL_DIR = (parameters or {}).get("rec_model_dir")
print(f"[paddleocr_engine] lang={LANG} use_angle_cls={USE_ANGLE_CLS}")

In [ ]:
def build_paddleocr_engine(lang: str = LANG, use_angle_cls: bool = USE_ANGLE_CLS):
    """Build a PaddleOCR instance lazily.

    The first call downloads the detection + recognition models; subsequent
    calls reuse the cached weights under ``~/.paddleocr/``.
    """
    from paddleocr import PaddleOCR  # imported here so the rest of the
                                     # portal doesn't need paddleocr at import time
    kwargs = {
        "lang": lang,
        "use_angle_cls": use_angle_cls,
        "show_log": False,
    }
    if DET_MODEL_DIR:
        kwargs["det_model_dir"] = DET_MODEL_DIR
    if REC_MODEL_DIR:
        kwargs["rec_model_dir"] = REC_MODEL_DIR
    return PaddleOCR(**kwargs)


def recognize(image, *, engine=None, lang: str = LANG, use_angle_cls: bool = USE_ANGLE_CLS):
    """Recognize text in ``image`` (BGR numpy array, file path, or bytes).

    Returns a list of ``(polygon, text, confidence)`` tuples in PaddleOCR's
    native order so the Python service can normalize the shape.
    """
    if engine is None:
        engine = build_paddleocr_engine(lang=lang, use_angle_cls=use_angle_cls)
    start = time.perf_counter()
    raw = engine.ocr(image, cls=use_angle_cls)
    elapsed = (time.perf_counter() - start) * 1000.0

    boxes: list[list] = []
    # raw shape: [ [ [box, (text, conf)], ... ] ]  (or [] if nothing found)
    if raw and raw[0]:
        for line in raw[0]:
            polygon, (text, conf) = line
            boxes.append([polygon, text, float(conf)])
    return {"boxes": boxes, "elapsed_ms": elapsed, "engine": "paddleocr"}


engine = build_paddleocr_engine()  # eagerly build so the first call is fast
print(f"[paddleocr_engine] ready lang={LANG}")